<a href="https://colab.research.google.com/github/jones14115/avatarify-python/blob/master/avatarify.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Avatarify Colab Server

This Colab notebook is for running Avatarify rendering server. It allows you to run Avatarify on your computer **without GPU** in this way:

1. When this notebook is executed, it starts listening for incoming requests from your computer;
1. You start the client on your computer and it connects to the notebook and starts sending requests;
1. This notebooks receives the requests from your computer, renders avatar images and sends them back;

To this end, all the heavy work is offloaded from your computer to this notebook so you don't need to have a beafy hardware on your PC anymore.


## Start the server
Run the cells below (Shift+Enter) sequentially and pay attention to the hints and instructions included in this notebook.

At the end you will get a command for running the client on your computer.

## Start the client

Make sure you have installed the latest version of Avatarify on your computer. Refer to the [README](https://github.com/alievk/avatarify#install) for the instructions.

When it's ready execute this notebook and get the command for running the client on your computer.


### Technical details

The client on your computer connects to the server via `ngrok` TCP tunnel or a reverse `ssh` tunnel.

`ngrok`, while easy to use, can induce a considerable network lag ranging from dozens of milliseconds to a second. This can lead to a poor experience.

A more stable connection could be established using a reverse `ssh` tunnel to a host with a public IP, like an AWS `t3.micro` (free) instance. This notebook provides a script for creating a tunnel, but launching an instance in a cloud is on your own (find the manual below).

# Install

### Avatarify
Follow the steps below to clone Avatarify and install the dependencies.

In [ ]:
# Cell 1: Clean environment
%cd /content
!rm -rf *

In [ ]:
# Cell 2: Clone repositories
!git clone https://github.com/alievk/avatarify.git
%cd avatarify
!git clone https://github.com/alievk/first-order-model.git fomm

In [ ]:
# Cell 3: Install dependencies
!pip install -q face-alignment==1.0.0 msgpack_numpy pyyaml==6.0.3
!pip install -q pyngrok

In [ ]:
# Cell 4: Download model weights
!scripts/download_data.sh

In [ ]:
# Cell 5: Setup ports and functions
import subprocess
import shlex
import json
import time
import re
from pyngrok import ngrok

# Input and output ports for communication
local_in_port = 5557
local_out_port = 5558

def get_tunnel_adresses_from_log(log_file='/tmp/ngrok.log', timeout=10):
    """
    Parse tunnel URLs from ngrok log file instead of HTTP API.
    This works with console_ui: False
    """
    start_time = time.time()
    in_addr = None
    out_addr = None

    while time.time() - start_time < timeout:
        try:
            with open(log_file, 'r') as f:
                content = f.read()

            # Find all tunnel URLs in the log
            # Pattern: url=tcp://xxxx.ngrok.io:xxxxx
            tunnels = re.findall(r'url=(tcp://[^\s]+)', content)

            for url in tunnels:
                # Determine which port this tunnel maps to by checking subsequent lines
                # Look for "addr=localhost:PORT" after the URL
                url_pattern = re.escape(url)
                port_match = re.search(
                    rf'{url_pattern}.*?addr=localhost:(\d+)',
                    content,
                    re.DOTALL
                )

                if port_match:
                    mapped_port = int(port_match.group(1))
                    print(f'{url} -> {mapped_port}')

                    if mapped_port == local_in_port:
                        in_addr = url
                        print(f'  [input tunnel found]')
                    elif mapped_port == local_out_port:
                        out_addr = url
                        print(f'  [output tunnel found]')

            if in_addr and out_addr:
                return in_addr, out_addr

        except FileNotFoundError:
            pass

        time.sleep(0.5)

    raise RuntimeError(f"Could not find both tunnels after {timeout}s. Check {log_file}")

def get_tunnel_adresses_from_pyngrok():
    """
    Alternative: Use pyngrok to get tunnels directly (no log parsing needed)
    """
    tunnels = ngrok.get_tunnels()
    in_addr = None
    out_addr = None

    for tunnel in tunnels:
        url = tunnel.public_url
        # Extract local port from tunnel config
        local_addr = tunnel.config.get('addr', '')
        port_match = re.search(r':(\d+)$', local_addr)

        if port_match:
            port = int(port_match.group(1))
            print(f'{url} -> {port} [{tunnel.name}]')

            if port == local_in_port:
                in_addr = url
            elif port == local_out_port:
                out_addr = url

    if not in_addr or not out_addr:
        raise RuntimeError(f"Missing tunnels: input={in_addr}, output={out_addr}")

    return in_addr, out_addr

In [ ]:
# Cell 6: Start the worker
print("Starting Avatarify worker...")

with open('/tmp/run.txt', 'w') as f:
    ps = subprocess.Popen(
        shlex.split(f'./run.sh --is-worker --in-port {local_in_port} --out-port {local_out_port} --no-vcam --no-conda'),
        stdout=f,
        stderr=f
    )
    time.sleep(3)

# Check if worker started
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started successfully"; fi

In [ ]:
# Cell 7: Configure ngrok (with console_ui: False as per your diff)
# Paste your authtoken here in quotes
authtoken = "YOUR_NGROK_AUTHTOKEN_HERE"  # ← Replace this

# Set your region: us, eu, ap, au, sa, jp, in
region = "eu"

config = f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
log: /tmp/ngrok.log
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
    f.write(config)

print("ngrok config written with console_ui: False")

In [ ]:
# Cell 8: Open tunnel and get addresses (FIXED METHOD)
import os
import signal

# Kill any existing ngrok processes
!pkill -f ngrok 2>/dev/null
time.sleep(1)

# Clear old log file
!rm -f /tmp/ngrok.log

# Start ngrok with the config
print("Starting ngrok tunnel...")
ngrok_ps = subprocess.Popen(
    shlex.split('./ngrok start --config ngrok.conf input output'),
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

# Wait for ngrok to initialize
time.sleep(4)

# Try to get tunnel addresses using the FIXED method
try:
    # Method 1: Parse from log file (works with console_ui: False)
    in_addr, out_addr = get_tunnel_adresses_from_log()
    print("\n✓ Tunnels discovered from log file")
except Exception as e:
    print(f"Log parsing failed: {e}")
    print("Trying alternative method...")

    # Method 2: Use pyngrok API (also works with console_ui: False)
    try:
        in_addr, out_addr = get_tunnel_adresses_from_pyngrok()
        print("\n✓ Tunnels discovered via pyngrok")
    except Exception as e2:
        print(f"Both methods failed: {e2}")
        print("Check ngrok status manually:")
        !cat /tmp/ngrok.log 2>/dev/null || echo "No log file yet"
        raise

print(f"\n{'='*60}")
print(f"INPUT TUNNEL:  {in_addr}")
print(f"OUTPUT TUNNEL: {out_addr}")
print(f"{'='*60}")

In [ ]:
# Cell 9: Print client commands
print('Copy-paste to your local terminal:\n')

print('Mac:')
print(f'./run_mac.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nWindows:')
print(f'run_windows.bat --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nLinux:')
print(f'./run.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')

### ngrok
Follow the steps below to setup ngrok. You will also need to sign up on the ngrok site and get your authtoken (free).


In [ ]:
# Download ngrok
!scripts/get_ngrok.sh

# Run
Start here if the runtime was restarted after installation.

In [ ]:
cd /content/avatarify

In [ ]:
#!git pull origin

In [ ]:
from subprocess import Popen, PIPE
import shlex
import json
import time


def run_with_pipe(command):
  commands = list(map(shlex.split,command.split("|")))
  ps = Popen(commands[0], stdout=PIPE, stderr=PIPE)
  for command in commands[1:]:
    ps = Popen(command, stdin=ps.stdout, stdout=PIPE, stderr=PIPE)
  return ps.stdout.readlines()


def get_tunnel_adresses():
  info = run_with_pipe("curl http://localhost:4040/api/tunnels")
  assert info

  info = json.loads(info[0])
  for tunnel in info['tunnels']:
    url = tunnel['public_url']
    port = url.split(':')[-1]
    local_port = tunnel['config']['addr'].split(':')[-1]
    print(f'{url} -> {local_port} [{tunnel["name"]}]')
    if tunnel['name'] == 'input':
      in_addr = url
    elif tunnel['name'] == 'output':
      out_addr = url
    else:
      print(f'unknown tunnel: {tunnel["name"]}')

  return in_addr, out_addr

In [ ]:
# Input and output ports for communication
local_in_port = 5557
local_out_port = 5558

# Start the worker


In [ ]:
# (Re)Start the worker
with open('/tmp/run.txt', 'w') as f:
  ps = Popen(
      shlex.split(f'./run.sh --is-worker --in-port {local_in_port} --out-port {local_out_port} --no-vcam --no-conda'),
      stdout=f, stderr=f)
  time.sleep(3)

This command should print lines if the worker is successfully started

In [ ]:
!ps aux | grep 'python3 afy/cam_fomm.py' | grep -v grep | tee /tmp/ps_run
!if [[ $(cat /tmp/ps_run | wc -l) == "0" ]]; then echo "Worker failed to start"; cat /tmp/run.txt; else echo "Worker started"; fi

# Open ngrok tunnel

#### Get ngrok token
Go to https://dashboard.ngrok.com/auth/your-authtoken (sign up if required), copy your authtoken and put it below.

In [ ]:
# Paste your authtoken here in quotes
authtoken = "1cBzFFwzSlaLhlRPXIHJiVLqtiQ_2cVsonJXe52B6DDyp8su7"

Set your region

Code | Region
--- | ---
us | United States
eu | Europe
ap | Asia/Pacific
au | Australia
sa | South America
jp | Japan
in | India

In [ ]:
# Set your region here in quotes
region = "eu"

In [ ]:
config =\
f"""
version: 2
authtoken: {authtoken}
region: {region}
console_ui: False
tunnels:
  input:
    addr: {local_in_port}
    proto: tcp
  output:
    addr: {local_out_port}
    proto: tcp
"""

with open('ngrok.conf', 'w') as f:
  f.write(config)

In [ ]:
# (Re)Open tunnel
ps = Popen('./scripts/open_tunnel_ngrok.sh', stdout=PIPE, stderr=PIPE)
time.sleep(3)

In [ ]:
# Get tunnel addresses
try:
  in_addr, out_addr = get_tunnel_adresses()
  print("Tunnel opened")
except Exception as e:
  [print(l.decode(), end='') for l in ps.stdout.readlines()]
  print("Something went wrong, reopen the tunnel")

### [Optional] AWS proxy
Alternatively you can create a ssh reverse tunnel to an AWS `t3.micro` instance (it's free). It has lower latency than ngrok.

1. In your AWS console go to Services -> EC2 -> Instances -> Launch Instance;
1. Choose `Ubuntu Server 18.04 LTS` AMI;
1. Choose `t3.micro` instance type and press Review and launch;
1. Confirm your key pair and press Launch instances;
1. Go to the security group of this instance and edit inbound rules. Add TCP ports 5557 and 5558 and set Source to Anywhere. Press Save rules;
1. ssh into the instance (you can find the command in the Instances if you click on the Connect button) and add this line in the end of `/etc/ssh/sshd_config`:
```
GatewayPorts yes
```
then restart `sshd`
```
sudo service sshd restart
```
1. Copy your `key_pair.pem` by dragging and dropping it into avatarify folder in this notebook;
1. Use the command below to open the tunnel;
1. Start client with a command (substitute `run_mac.sh` with `run_windows.bat` or `run.sh`)
```
./run_mac.sh --is-client --in-addr tcp://instace.compute.amazonaws.com:5557 --out-addr tcp://instance.compute.amazonaws.com:5558
```

In [ ]:
# Open reverse ssh tunnel (uncomment line below)
# !./scripts/open_tunnel_ssh.sh key_pair.pem ubuntu@instance.compute.amazonaws.com

# Start the client
When you run the cell below it will print a command. Run this command on your computer:

1. Open a terminal (in Windows open `Anaconda Prompt`);
2. Change working directory to the `avatarify` directory:</br>
* Windows (change `C:\path\to\avatarify` to your path)</br>
`cd C:\path\to\avatarify`</br></br>
* Mac/Linux (change `/path/to/avatarify` to your path)</br>
`cd /path/to/avatarify`
3. Copy-paste to the terminal the command below and run;
4. It can take some time to connect (usually up to 10 seconds). If the preview window doesn't appear in a minute or two, look for the errors above in this notebook and report in the [issues](https://github.com/alievk/avatarify/issues) or [Slack](https://join.slack.com/t/avatarify/shared_invite/zt-dyoqy8tc-~4U2ObQ6WoxuwSaWKKVOgg).

In [ ]:
print('Copy-paste to the terminal the command below and run (press Enter)\n')
print('Mac:')
print(f'./run_mac.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nWindows:')
print(f'run_windows.bat --is-client --in-addr {in_addr} --out-addr {out_addr}')
print('\nLinux:')
print(f'./run.sh --is-client --in-addr {in_addr} --out-addr {out_addr}')

# Logs

If something doesn't work as expected, please run the cells below and include the logs in your report.

In [ ]:
#@title
!cat ./var/log/cam_fomm.log | head -100

In [ ]:
#@title
!cat ./var/log/recv_worker.log | tail -100

In [ ]:
#@title
!cat ./var/log/predictor_worker.log | tail -100

In [ ]:
#@title
!cat ./var/log/send_worker.log | tail -100